# 🕵️ Case file 04 — The booking lookup that keeps failing

> **Mission briefing.** Last call of the day. Customers give valid booking references, but the booking lookup lets them down: first it fails, then it answers about the wrong trip, then it becomes painfully slow. The assistant itself is fine. Its **tool** (the booking lookup it calls) is not. Open the activity logs, find what the tool really did, and fix the tool, not the assistant's wording.

Each run leaves an **activity log** (Langfuse calls it a *trace*), made of **steps** (*spans*). The `get_booking` step shows the details the assistant sent to the tool, what the tool returned, any error, and how long it took.

## Case objective
Use the tool step's input, output, error status and timing to find and fix three different tool problems.

## What to expect
Three levels. Each run prints the assistant's answer, how long it took, and a link to the activity log. What the tool actually did is only in the activity log.

## Bad practice
Rewording the assistant's instructions when the real problem is in the tool it depends on.

## Good practice
Open the tool step and compare what was asked, what was returned and how long it took. Then fix the tool.

## Why the good practice works
The tool step records the exact handover between the assistant and the tool, so the cause is visible even when the final answer hides it.

## Deliverable
A claim code for each level you solve, and for your demo, a before/after pair of activity-log links for one fixed tool.

## 🎮 How the game works

- Put your team name in the next cell and run it. Use **the same name in every notebook**.
- Each level asks you to find something in the activity logs. Type your answer in `check_flag(...)` and run it. Wrong guesses cost nothing.
- Correct answers give a **claim code**. Paste it into the game master's form.
- Stuck? `hint("4-1")` shows a hint. Each hint costs 25% of that level's points (two hints at most).
- Points: Level 1 = 100, Level 2 = 200, Level 3 = 300.

In [1]:
import time

from experiment_utils import (BOOKINGS, check_flag, close_case, get_booking_l1, get_booking_l2, hint,
                              make_partner_lookup, run_tool_loop, set_team, show_tool_results)

TEAM = "your-team-name"  # ← your team name, the same in every notebook
set_team(TEAM)

⚠️  TEAM is still 'your-team-name'. Claim codes will not score: change it and rerun this cell.


'your-team-name'

## Level 1 — The wrong label (100 points)

A customer asks about booking SKY-1042. The assistant calls the lookup tool and something goes wrong.

**Find:** the name of the detail the tool tried to read. Then complete `my_get_booking` so it reads the detail the assistant actually sends, and rerun.

In [2]:
question = "What is the status of booking SKY-1042?"
level1 = [run_tool_loop(question, "broken", lookup=get_booking_l1)]
show_tool_results(level1)


level 1 · broken
  answer: I'm sorry — I wasn't able to look up your booking just now. Please try again later.
  latency: 2.4s
  activity log: http://localhost:3000/project/langfuse-observability/traces/35b7645a0e5385888f7c26ce39f1523e


In [3]:
check_flag("4-1", "type your answer here")
# hint("4-1")

❌ Not quite. Try again, or run hint("4-1").


In [4]:
def my_get_booking(**details):
    booking_ref = details["?"]  # ← which detail does the assistant send? Check the tool step's input.
    return BOOKINGS[booking_ref]

level1.append(run_tool_loop(question, "fixed", lookup=my_get_booking))
show_tool_results(level1[-1:])


level 1 · fixed
  answer: I'm sorry — I wasn't able to look up your booking just now. Please try again later.
  latency: 2.2s
  activity log: http://localhost:3000/project/langfuse-observability/traces/35383edf1772b77b70440a5a26e6d531


## Level 2 — The wrong trip (200 points)

The IT team shipped a "more forgiving" lookup that never fails, whatever the spelling of the reference. A customer asks about booking **SKY-2087**. There is no error this time, but read the answer carefully.

**Find:** the booking reference the tool actually returned. Then fix `my_get_booking_v2` so it returns exactly the booking asked for, or a clear "not found", and rerun.

In [5]:
question2 = "What is the status of booking SKY-2087?"
level2 = [run_tool_loop(question2, "forgiving", level=2, lookup=get_booking_l2)]
show_tool_results(level2)


level 2 · forgiving
  answer: Your booking is confirmed. It's a premium-cabin trip from Madrid to Rome on 15 January 2027 for 3 passengers.
  latency: 3.9s
  activity log: http://localhost:3000/project/langfuse-observability/traces/cae1b39573565fe8af7729d224370aed


In [6]:
check_flag("4-2", "type your answer here")
# hint("4-2")

❌ Not quite. Try again, or run hint("4-2").


In [7]:
def my_get_booking_v2(**details):
    booking_ref = details["booking_ref"]
    # ← return the booking for exactly this reference, or {"error": "booking not found"}.
    #   Never return someone else's booking.
    return get_booking_l2(**details)

level2.append(run_tool_loop(question2, "fixed", level=2, lookup=my_get_booking_v2))
show_tool_results(level2[-1:])


level 2 · fixed
  answer: Your booking is confirmed. It's a premium-cabin trip from Madrid to Rome on 15 January 2027 for 3 passengers.
  latency: 3.6s
  activity log: http://localhost:3000/project/langfuse-observability/traces/d3d9306d53a4b21b3647f05ca1dccee4


## Level 3 — The slow lookup (300 points)

The lookup now confirms every booking with a partner system. Confirmed bookings are kept in a fast memory (a *cache*), labelled by `cache_key`. Customers complain that every lookup takes ages. Sort the steps of the lookup by time in the activity log.

**Find:** the name of the slow step and how many times it runs per lookup, joined by a dash (for example `some-step-7`). Then fix `cache_key` so the cache is actually used, and confirm the lookup is fast again.

In [8]:
def cache_key(booking_ref):
    return f"{booking_ref}:{time.time()}"  # the label used to find a booking in the cache

level3 = [run_tool_loop(question, "slow", level=3, lookup=make_partner_lookup(cache_key))]
show_tool_results(level3)


level 3 · slow
  answer: Your booking from Lisbon to Madrid on 1 December 2026 is confirmed. It's for 2 passengers in economy.
  latency: 6.7s
  activity log: http://localhost:3000/project/langfuse-observability/traces/fb05d13989991680f276447fb75561f2


In [9]:
check_flag("4-3", "type your answer here")
# hint("4-3")

❌ Not quite. Try again, or run hint("4-3").


## 🏁 Wrap-up

Run the last cell to close the case and send the activity logs to Langfuse.

In [10]:
close_case("activity-4", level1 + level2 + level3)


case activity-4 closed — session activity-4-c7d3934c
level 1 · broken: http://localhost:3000/project/langfuse-observability/traces/35b7645a0e5385888f7c26ce39f1523e
level 1 · fixed: http://localhost:3000/project/langfuse-observability/traces/35383edf1772b77b70440a5a26e6d531
level 2 · forgiving: http://localhost:3000/project/langfuse-observability/traces/cae1b39573565fe8af7729d224370aed
level 2 · fixed: http://localhost:3000/project/langfuse-observability/traces/d3d9306d53a4b21b3647f05ca1dccee4
level 3 · slow: http://localhost:3000/project/langfuse-observability/traces/fb05d13989991680f276447fb75561f2
activity logs flushed to Langfuse
